# 01 EDA: VN1 after cleaning

Explores the outputs of `python -m src.run_cleaning`. Uses Phase 0 (training) only; Phase 2 is never loaded here.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, matplotlib.pyplot as plt
from src.config import get_config

P = ROOT / 'data' / 'processed'
weekly = pd.read_parquet(P / 'phase0_clean.parquet')
series = pd.read_parquet(P / 'series_table.parquet')
weekly.shape, series.shape

## Flag rates

In [ ]:
flags = ['identical_run', 'is_smoothed', 'is_smoothed_era', 'price_zero', 'price_filled', 'price_shift', 'is_suspected_stockout']
pd.DataFrame({'weeks': weekly[flags].sum(), 'share': weekly[flags].mean().round(4)})

In [ ]:
series[['is_new', 'is_inactive', 'has_price_shift', 'has_suspected_stockout']].mean().round(3)

## Demand types: full history vs weekly era only

In [ ]:
pd.crosstab(series['demand_type'], series['demand_type_weekly_era'], margins=True)

## Look at one series
Change `uid` to any id from `series`.

In [ ]:
uid = series.sort_values('volume_rank')['unique_id'].iloc[0]
d = weekly[weekly.unique_id == uid]
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].plot(d.ds, d.y)
axes[0].scatter(d.ds[d.is_smoothed], d.y[d.is_smoothed], s=8, color='tab:orange', label='is_smoothed')
axes[0].legend(); axes[0].set_title(f'{uid}: units')
axes[1].plot(d.ds, d.relative_price); axes[1].axhline(1, color='grey', lw=0.8); axes[1].set_title('relative_price')
plt.tight_layout()

## Repeat rate by week (the smoothed-era signal)

In [ ]:
from src.cleaning import estimate_weekly_switch_date
sw = estimate_weekly_switch_date(weekly[['unique_id', 'ds', 'y']])
ax = sw['rate'].plot(figsize=(11, 3), title=f"switch {sw['switch_date'].date()}")
ax.axvline(sw['switch_date'], ls='--', color='grey');